# NB03 — Triton Fundamentals

**North star callback:** The fused operations that make Unsloth fast (flash attention,
fused RoPE, chunked cross-entropy) are all written in Triton. This notebook teaches you
to read and write Triton kernels from first principles.

**Mental model:** A Triton kernel is a tiled map over a tensor, with explicit control
over what lives in SRAM (fast) vs HBM (slow).

## 1. The programming model

Triton launches a **grid** of **programs**. Each program handles one **tile** of the data.
Inside a program, you work with **blocks** — contiguous ranges of indices.

```
Grid(N // BLOCK_SIZE programs)
└── Program i handles indices [i*BLOCK_SIZE : (i+1)*BLOCK_SIZE]
    └── Data is loaded into SRAM, computed on, written back to HBM
```

Unlike CUDA, you don't manage threads or warps explicitly. Triton handles that.
You only specify: what tile does each program handle? What computation happens on it?

In [1]:
import torch
import triton
import triton.language as tl

@triton.jit
def fused_scale_relu_kernel(
    x_ptr,       # pointer to input tensor
    out_ptr,     # pointer to output tensor
    scale,       # scalar multiplier
    n_elements,  # total number of elements
    BLOCK_SIZE: tl.constexpr,  # tile size — must be power of 2
):
    # Each program handles one tile
    pid = tl.program_id(axis=0)  # which tile am I?
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)

    # Mask: guard against out-of-bounds on the last tile
    mask = offsets < n_elements

    # Load from HBM into SRAM
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)

    # Compute (fused: scale + relu in SRAM, no intermediate HBM write)
    x = x * scale
    x = tl.where(x > 0, x, 0.0)  # ReLU

    # Write result back to HBM
    tl.store(out_ptr + offsets, x, mask=mask)


def fused_scale_relu(x: torch.Tensor, scale: float) -> torch.Tensor:
    out = torch.empty_like(x)
    n = x.numel()
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(n, BLOCK_SIZE),)  # number of programs to launch
    fused_scale_relu_kernel[grid](x, out, scale, n, BLOCK_SIZE=BLOCK_SIZE)
    return out


# Test correctness
x = torch.randn(1024 * 100, device="cuda", dtype=torch.float32)
out_triton = fused_scale_relu(x, scale=2.0)
out_ref = torch.relu(x * 2.0)
assert torch.allclose(out_triton, out_ref, atol=1e-5), "Mismatch!"
print("✓ fused_scale_relu matches reference")

✓ fused_scale_relu matches reference


In [2]:
@triton.autotune(
    configs=[
        triton.Config({"BLOCK_SIZE": 256}),
        triton.Config({"BLOCK_SIZE": 512}),
        triton.Config({"BLOCK_SIZE": 1024}),
        triton.Config({"BLOCK_SIZE": 2048}),
        triton.Config({"BLOCK_SIZE": 4096}),
    ],
    key=["n_elements"],
)
@triton.jit
def fused_scale_relu_autotuned(
    x_ptr, out_ptr, scale, n_elements, BLOCK_SIZE: tl.constexpr
):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    tl.store(out_ptr + offsets, tl.where(x * scale > 0, x * scale, 0.0), mask=mask)


def bench_and_compare(size: int = 1024 * 1024 * 16):
    x = torch.randn(size, device="cuda", dtype=torch.float32)
    ms_triton = triton.testing.do_bench(lambda: fused_scale_relu(x, 2.0))
    ms_torch = triton.testing.do_bench(lambda: torch.relu(x * 2.0))
    bw_triton = 2 * size * 4 / ms_triton * 1e-6  # GB/s (read + write, float32)
    bw_torch = 2 * size * 4 / ms_torch * 1e-6
    print(f"Triton: {ms_triton:.3f} ms  ({bw_triton:.0f} GB/s)")
    print(f"PyTorch:{ms_torch:.3f} ms  ({bw_torch:.0f} GB/s)")
    print(f"(Note: PyTorch may also fuse these — the win is explicit control)")

bench_and_compare()

Triton: 0.171 ms  (787 GB/s)
PyTorch:0.330 ms  (406 GB/s)
(Note: PyTorch may also fuse these — the win is explicit control)


## 2. Writing a parallel reduction

Reductions (sum, max, softmax numerator) are the building block of attention.
A naive Python loop is O(N). A parallel tree reduction is O(log N).
Triton handles the inter-warp communication for us.

In [3]:
@triton.jit
def sum_kernel(
    x_ptr,
    out_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    """Each program reduces one block to a scalar, then atomic-adds to output."""
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    block_sum = tl.sum(x, axis=0)  # tree reduction within the block
    tl.atomic_add(out_ptr, block_sum)


def triton_sum(x: torch.Tensor) -> torch.Tensor:
    out = torch.zeros(1, device=x.device, dtype=x.dtype)
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(x.numel(), BLOCK_SIZE),)
    sum_kernel[grid](x, out, x.numel(), BLOCK_SIZE=BLOCK_SIZE)
    return out

x = torch.randn(1024 * 1024, device="cuda")
ref = x.sum()
got = triton_sum(x)
assert torch.allclose(got, ref, atol=1e-2), f"Got {got.item():.4f}, expected {ref.item():.4f}"
print(f"✓ triton_sum: {got.item():.4f}  ref: {ref.item():.4f}")

✓ triton_sum: -477.9868  ref: -477.9865


In [4]:
import sys; sys.path.insert(0, "..")
from utils.benchmark import compare

size = 1024 * 1024 * 32
x = torch.randn(size, device="cuda")

results = compare(
    fns={
        "torch_sum": lambda: x.sum(),
        "triton_sum": lambda: triton_sum(x),
    },
    notebook="nb03",
    experiment="reduction_benchmark",
    n_warmup=10,
    n_repeat=50,
)
for label, r in results.items():
    print(f"{label}: {r.latency_ms:.3f} ms")

torch_sum: 0.146 ms
triton_sum: 0.147 ms


## 3. Exercises

1. **Change BLOCK_SIZE** in `fused_scale_relu_kernel` from 1024 to 256 and 4096.
   Run `bench_and_compare()` each time. What happens to throughput and why?

2. **Add a bias term**: modify `fused_scale_relu_kernel` to accept a `bias` pointer
   and add it to `x` before the ReLU. Verify correctness against `torch.relu(x * scale + bias)`.

3. **Implement parallel max**: write a `max_kernel` analogous to `sum_kernel`
   using `tl.max` instead of `tl.sum` and `tl.atomic_max`. Verify against `x.max()`.

**Next:** NB04 uses these primitives to build flash attention from scratch.